<a href="https://colab.research.google.com/github/joobnsucky/MyAI/blob/main/%EC%8B%A4%EC%8A%B5%EA%B2%B0%EA%B3%BC%EC%A0%95%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 실습 결과 정리
아래 셀들은 실습 노트북 파일의 가장 마지막에 덧붙여 실행할 수 있도록 작성되었습니다.

#### 1. Smoothing 적용 전후에 미관측 bigram의 확률을 비교하세요.

In [ ]:
context = "like"
probe_word = "good" # 실습 말뭉치에서 'like good'이라는 순서로 등장한 적이 없음을 가정합니다.

prob_before = next_word_probs(context, alpha=0.0)[vocabulary.index(probe_word)]
prob_after = next_word_probs(context, alpha=0.1)[vocabulary.index(probe_word)]

print(f"Smoothing 전 (alpha=0.0) P({probe_word} | {context}) = {prob_before:.4f}")
print(f"Smoothing 후 (alpha=0.1) P({probe_word} | {context}) = {prob_after:.4f}")

**답변:**
Smoothing을 적용하기 전(`alpha=0.0`)에는 말뭉치에 한 번도 등장하지 않은 미관측 문맥 조합(`like` 뒤에 `good`이 오는 경우)의 확률이 0으로 계산됩니다. 하지만 Add-alpha Smoothing을 적용한 후(`alpha=0.1`)에는 0이 아닌 양의 확률을 부여받게 되어, 한 번도 보지 못한 문장 조합이 등장했을 때 전체 문장 확률이 0이 되어버리는 희소성(Sparsity) 문제를 방지할 수 있습니다.

#### 2. Skip-gram에서 window 크기 두 가지를 사용해 동일 단어의 상위 이웃을 비교하세요.

In [ ]:
query_word = "cat"

# window=1과 window=3에서의 학습 결과를 비교합니다.
my_vectors_w1, _ = train_skipgram(window=1)
my_vectors_w3, _ = train_skipgram(window=3)

print(f"window=1 | {query_word} ->", similar_words(query_word, my_vectors_w1))
print(f"window=3 | {query_word} ->", similar_words(query_word, my_vectors_w3))

**답변:**
Window 크기가 작을 때(`window=1`)는 단어 바로 옆에 인접한 단어들이 학습되므로 문법적 기능이나 직접적인 수식 관계가 임베딩에 강하게 반영됩니다. 반면 Window 크기를 늘릴 경우(`window=3`), 더 넓은 범위의 문장 전체 문맥을 포괄하게 되어, 해당 단어가 주로 쓰이는 주제나 의미(예: 동물, 애완동물)가 유사한 단어들이 상위 이웃으로 추출되는 경향을 확인할 수 있습니다.

#### 3. 리뷰 분류 모델의 검증 정확도와 파라미터 수를 비교하고, 결과를 한 문장으로 해석하세요.

In [ ]:
# 이전 실습 3에서 학습된 histories 딕셔너리를 활용합니다.
for name, history in histories.items():
    if name == 'Embedding + Mean':
        params = build_sentiment_model(use_rnn=False).count_params()
    else:
        params = build_sentiment_model(use_rnn=True).count_params()

    val_acc = history.history['val_accuracy'][-1]
    print(f"{name:25s} | 파라미터 수={params:,} | 마지막 검증 정확도={val_acc:.1%}")

**답변:**
순서 정보를 처리하는 `SimpleRNN` 모델은 단순 `Mean` 풀링 모델보다 훨씬 많은 파라미터를 요구하지만, 영화 리뷰 감성 분류 작업의 경우 복잡한 어순이나 문맥의 순서보다 'good', 'boring'과 같은 몇몇 핵심 감성 단어의 존재 유무가 더 중요하게 작용하기 때문에 단순 평균(Mean) 모델만으로도 유사하거나 더 나은 분류 성능을 낼 수 있습니다.